# Model Training and Evaluation for Loan Default Prediction

This notebook trains and compares three classification models while using imbalance-aware metrics. Accuracy is not used as the primary evaluation metric because the target is imbalanced.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
df = pd.read_csv('../outputs/cleaned_loan_default.csv')
df.head()

In [ ]:
target_col = 'Default'
id_col = 'LoanID'

X = df.drop(columns=[id_col, target_col])
y = df[target_col]

# Keep the useful and non-identifier features used during feature engineering
numeric_features = [
    'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
    'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'
]

binary_features = ['HasMortgage', 'HasDependents', 'HasCoSigner']
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

selected_features = numeric_features + binary_features + categorical_features
X = X[selected_features]
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts(normalize=True).sort_index()}')
print(f'y_test distribution:\n{y_test.value_counts(normalize=True).sort_index()}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('bin', OneHotEncoder(handle_unknown='ignore'), binary_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ],
    remainder='drop',
    sparse_threshold=0.0,
)

preprocessor.fit(X_train)
X_train_prepared = preprocessor.transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f'Prepared training features shape: {X_train_prepared.shape}')
print(f'Prepared test features shape: {X_test_prepared.shape}')

In [1]:
from sklearn.base import clone

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=2000,
        class_weight='balanced',
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced',
        min_samples_leaf=2,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42,
    ),
}

results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC-AUC': roc_auc,
        'Confusion Matrix': np.array([[tn, fp], [fn, tp]]),
    })

    print(f'\n=== {name} ===')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'ROC-AUC: {roc_auc:.4f}')
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df[[
    'Model', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Confusion Matrix'
]]
comparison_df

In [ ]:
# Ranking by the most important imbalance-aware metrics
best_model = comparison_df.sort_values(
    by=['F1-score', 'ROC-AUC', 'Recall', 'Precision'],
    ascending=False,
).iloc[0]

print(f"\nStrongest model based on F1-score, ROC-AUC, Recall, and Precision: {best_model['Model']}")
print(best_model.to_string())

In [ ]:
import pickle
from pathlib import Path

selected_model_name = best_model['Model']
selected_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', clone(models[selected_model_name])),
])
selected_pipeline.fit(X_train, y_train)
selected_pipeline.comparison_df = comparison_df.drop(columns=['Confusion Matrix'])
selected_pipeline.selected_model_name = selected_model_name
model_output_path = Path('../outputs/credit_risk_model.pkl')
with model_output_path.open('wb') as model_file:
    pickle.dump(selected_pipeline, model_file, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved selected pipeline to {model_output_path}')


## Model comparison summary
The comparison table above evaluates the models on precision, recall, F1-score, and ROC-AUC. These are more informative than accuracy for an imbalanced Default target. The model with the strongest overall metrics is identified from the ranked results.